In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder

# ====================================================================
# ☢️ PROTOCOLE : "POISSON NUCLEAR"
# Stratégie : Poisson Loss + Sniper Features + 10-Fold Bagging
# ====================================================================

# 1. CHARGEMENT & FEATURE ENGINEERING
# --------------------------------------------------------------------
print("⏳ Chargement des munitions...")
train_data = pd.read_csv('conversion_data_train.csv')
test_data = pd.read_csv('conversion_data_test.csv')

def feature_engineering(df):
    df_eng = df.copy()
    # Interactions stratégiques (Sniper Features)
    # L'arbre va adorer ces ratios pour isoler les comportements intenses
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    return df_eng

# Application sur le brut (on garde le bruit et les doublons !)
X = feature_engineering(train_data.drop('converted', axis=1))
y = train_data['converted']
X_test = feature_engineering(test_data)

# 2. ENCODAGE ROBUSTE
# --------------------------------------------------------------------
# On encode sur l'ensemble (Train + Test) pour ne rater aucune catégorie
categorical_cols = ['country', 'source']
cat_indices = [X.columns.get_loc(col) for col in categorical_cols]

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined)
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

# 3. LA STRUCTURE NUCLÉAIRE : 10-Fold Stratified CV
# --------------------------------------------------------------------
n_folds = 10
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Tableaux de stockage
oof_preds = np.zeros(len(X))          # Prédictions sur le train (pour trouver le seuil parfait)
test_preds_accumulated = np.zeros(len(X_test)) # Cumul des votes pour le test
fold_scores = []

print(f"🚀 Lancement de la séquence de tir ({n_folds} Folds)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    # L'Arme : HistGradientBoosting avec Perte POISSON
    # C'est lui qui fait la différence sur les 3% de conversion
    model = HistGradientBoostingRegressor(
        loss='poisson',           # La clé du succès
        learning_rate=0.05,
        max_iter=600,             
        max_depth=8,
        l2_regularization=0.1,
        categorical_features=cat_indices,
        random_state=42 + fold    # Variation de graine pour diversifier les avis
    )
    
    model.fit(X_train_fold, y_train_fold)
    
    # Prédictions (Intensités)
    val_preds = model.predict(X_val_fold)
    oof_preds[val_idx] = val_preds
    
    # Accumulation des tirs sur le Test Set
    test_preds_accumulated += model.predict(X_test)
    
    # Score du pli
    auc_fold = roc_auc_score(y_val_fold, val_preds)
    fold_scores.append(auc_fold)
    print(f"  > Fold {fold+1}/{n_folds} | ROC-AUC: {auc_fold:.5f}")

# 4. FINALISATION & OPTIMISATION
# --------------------------------------------------------------------
# Moyenne des 10 modèles (Bagging)
test_preds_avg = test_preds_accumulated / n_folds

print("\n🔍 Optimisation du Seuil sur l'ensemble des données OOF...")
best_f1 = 0
best_threshold = 0
# On scanne les intensités prédites pour trouver le point de bascule F1 max
thresholds = np.linspace(oof_preds.min(), oof_preds.max(), 500)

for t in thresholds:
    preds_binary = (oof_preds >= t).astype(int)
    score = f1_score(y, preds_binary)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

# 5. RAPPORT DE MISSION
# --------------------------------------------------------------------
print(f"\n💎 RÉSULTATS POISSON NUCLEAR :")
print(f"ROC-AUC Moyen (Stabilité) : {np.mean(fold_scores):.5f}")
print(f"Meilleur F1 OOF (Réalisme): {best_f1:.5f}")
print(f"Seuil Optimal Détecté     : {best_threshold:.6f}")

# Génération du fichier
final_test_binary = (test_preds_avg >= best_threshold).astype(int)
submission = pd.DataFrame({'converted': final_test_binary})
submission.to_csv('submission_POISSON_NUCLEAR.csv', index=False)

print(f"\n✅ Cible verrouillée. Fichier 'submission_POISSON_NUCLEAR.csv' généré avec {final_test_binary.sum()} conversions.")

⏳ Chargement des munitions...
🚀 Lancement de la séquence de tir (10 Folds)...
  > Fold 1/10 | ROC-AUC: 0.98676
  > Fold 2/10 | ROC-AUC: 0.98361
  > Fold 3/10 | ROC-AUC: 0.98338
  > Fold 4/10 | ROC-AUC: 0.98612
  > Fold 5/10 | ROC-AUC: 0.98753
  > Fold 6/10 | ROC-AUC: 0.98433
  > Fold 7/10 | ROC-AUC: 0.98278
  > Fold 8/10 | ROC-AUC: 0.98504
  > Fold 9/10 | ROC-AUC: 0.98784
  > Fold 10/10 | ROC-AUC: 0.98576

🔍 Optimisation du Seuil sur l'ensemble des données OOF...

💎 RÉSULTATS POISSON NUCLEAR :
ROC-AUC Moyen (Stabilité) : 0.98531
Meilleur F1 OOF (Réalisme): 0.76807
Seuil Optimal Détecté     : 0.379332

✅ Cible verrouillée. Fichier 'submission_POISSON_NUCLEAR.csv' généré avec 909 conversions.
